In [3]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME')

connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(connection_string)

conn = engine.connect()
print("Connection réussie")
conn.close()

Connection réussie


In [4]:
df = pd.read_csv('financecore_clean.csv')
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (2000, 34)


,transaction_id,client_id,date_transaction,montant,devise,taux_change_eur,montant_eur,categorie,produit,agence,...,montant_eur_verifie,diff_eur,is_calc_error,categorie_risque,is_credit,amount_signed,solde_net,is_rejected,taux_rejet,taux_rejet_pct
0,TXN000559,CLI0060,2022-04-19 02:31:00,2050.42,EUR,1.00,2050.42,Depot especes,Compte Epargne,Marseille-Vieux-Port,...,2050.420000,0.000000,False,NaN,True,2050.42,21599.14,False,0.055728,5.572755
1,TXN001154,CLI0057,2024-06-20 20:51:00,-123.66,GBP,0.86,-143.79,Retrait DAB,Credit Consommation,NaN,...,-143.790698,0.000698,False,High,False,123.66,6995.71,True,NaN,NaN
2,TXN000764,CLI0015,2024-08-28 05:03:00,-396.17,EUR,1.00,-396.17,Prelevement,PEA,Lyon-Part-Dieu,...,-396.170000,0.000000,False,Medium,False,396.17,10808.05,False,0.040134,4.013378
3,TXN001598,CLI0045,2024-01-07 08:16:00,225.20,EUR,1.00,225.20,Paiement CB,Credit Consommation,Bordeaux-Meriadeck,...,225.200000,0.000000,False,Low,True,225.20,24235.30,False,0.045198,4.519774
4,TXN001873,CLI0034,2024-08-11 19:52:00,935.32,EUR,1.00,935.32,Interets,Credit Immobilier,Bordeaux-Meriadeck,...,935.320000,0.000000,False,High,True,935.32,11456.22,False,0.045198,4.519774


Séparer financecore_clean.csv selon les tables normalisées avant insertion
--Insérer les données

In [ ]:
import pandas as pd
from sqlalchemy import text

try:

    df = pd.read_csv('financecore_clean.csv')

    
    unique_segments = df['segment_client'].fillna('Inconnu').unique()
    segment_mapping = {name: i+1 for i, name in enumerate(unique_segments)}
    segments_df = pd.DataFrame([{'segment_id': id, 'nom_segment': name} for name, id in segment_mapping.items()])

  
    agences = df[['agence']].drop_duplicates().dropna().copy()
    agences['agence_id'] = range(1, len(agences) + 1)
    agences_to_sql = agences.rename(columns={'agence': 'nom_agence'})

    
    produits = df[['produit']].drop_duplicates().dropna().copy()
    produits['produit_id'] = range(1, len(produits) + 1)
    produits_to_sql = produits.rename(columns={'produit': 'type_produit'})

    clients = df[['client_id', 'segment_client', 'score_credit_client', 'categorie_risque']].drop_duplicates(subset=['client_id']).copy()
    clients = clients.rename(columns={'score_credit_client': 'score_credit'})                               
    clients['segment_id'] = clients['segment_client'].fillna('Inconnu').map(segment_mapping)
    clients['nom_client'] = None
    clients_to_sql = clients[['client_id', 'nom_client', 'score_credit', 'categorie_risque', 'segment_id']]


    df_enriched = df.merge(agences, on='agence', how='left')
    df_enriched = df_enriched.merge(produits, on='produit', how='left')

    comptes = df_enriched[['client_id', 'produit', 'solde_avant', 'date_formatted', 'agence_id', 'produit_id']].drop_duplicates(subset=['client_id']).copy()
    comptes['compte_id'] = comptes['client_id']
    comptes = comptes.rename(columns={'produit': 'type_compte'})
    comptes_to_sql = comptes[['compte_id', 'client_id', 'type_compte', 'solde_avant', 'date_formatted', 'agence_id', 'produit_id']]

  
    transactions = df[[
        'transaction_id', 'client_id', 'date_transaction', 'montant', 
        'devise', 'taux_change_eur', 'montant_eur', 'type_operation', 
        'statut', 'is_anomaly', 'is_outlier_montant', 'is_rejected'
    ]].copy()
    transactions = transactions.rename(columns={'client_id': 'compte_id'})
    transactions['date_transaction'] = pd.to_datetime(transactions['date_transaction'])

 
    with engine.connect() as connection:
        print(" Nettoyage des anciennes données...")
        connection.execute(text("TRUNCATE TABLE transactions, comptes, clients, agences, produits, segments CASCADE;"))
        connection.commit()

        print("Insertion des nouvelles données...")
        

        segments_df.to_sql('segments', con=engine, if_exists='append', index=False)
        agences_to_sql.to_sql('agences', con=engine, if_exists='append', index=False)
        produits_to_sql.to_sql('produits', con=engine, if_exists='append', index=False)
        clients_to_sql.to_sql('clients', con=engine, if_exists='append', index=False)
        comptes_to_sql.to_sql('comptes', con=engine, if_exists='append', index=False)
        transactions.to_sql('transactions', con=engine, if_exists='append', index=False)
        
        print("\n Tout est chargé. Vous pouvez maintenant vérifier vos vues sur pgAdmin.")

except Exception as e:
    print(f" Une erreur est survenue: {e}")

🧹 تنظيف البيانات القديمة...
📥 جاري إدخال البيانات الجديدة...

🚀 مبروك! كولشي عامر دابا. تقدري تمشي لـ pgAdmin وتشوفني الـ Views ديالك واش خدامين.


Implémenter la gestion des conflits (ON CONFLICT DO NOTHING / DO UPDATE)

In [6]:
from sqlalchemy import text

def handle_clients_conflicts(engine, clients_to_sql):
    """
    Gère les conflits d'insertion pour la table clients.
    Si un client existe déjà, ses informations (score et risque) sont mises à jour.
    """

    clients_to_sql.to_sql('temp_clients', con=engine, if_exists='replace', index=False)
    
    with engine.connect() as connection:
      
        upsert_query = text("""
            INSERT INTO clients (client_id, nom_client, score_credit, categorie_risque, segment_id)
            SELECT client_id, nom_client, score_credit, categorie_risque, segment_id FROM temp_clients
            ON CONFLICT (client_id) 
            DO UPDATE SET 
                score_credit = EXCLUDED.score_credit,
                categorie_risque = EXCLUDED.categorie_risque;
        """)
        connection.execute(upsert_query)
        
      
        connection.execute(text("DROP TABLE temp_clients;"))
        connection.commit()
        
    print("✅ Gestion des conflits terminée et données clients mises à jour avec succès.")

handle_clients_conflicts(engine, clients_to_sql)

✅ Gestion des conflits terminée et données clients mises à jour avec succès.


Vérifier l'intégrité référentielle après chargement (COUNT…..)

In [7]:
def check_database_integrity(engine):
    with engine.connect() as conn:
        print(" --- RAPPORT D'INTÉGRITÉ DE LA BASE DE DONNÉES ---")
        
       
        tables = ['segments', 'agences', 'produits', 'clients', 'comptes', 'transactions']
        for table in tables:
            count = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).fetchone()[0]
            print(f"✔️ Nombre de lignes dans '{table}': {count}")

        orphans = conn.execute(text("""
            SELECT COUNT(t.transaction_id) 
            FROM transactions t 
            LEFT JOIN comptes c ON t.compte_id = c.compte_id 
            WHERE c.compte_id IS NULL;
        """)).fetchone()[0]
        
        if orphans == 0:
            print("\n Sécurité : Toutes les transactions sont rattachées à des comptes existants.")
        else:
            print(f"\n ALERTE : Il existe {orphans} transaction(s) orpheline(s) (sans compte correspondant) !")

check_database_integrity(engine)

 --- RAPPORT D'INTÉGRITÉ DE LA BASE DE DONNÉES ---
✔️ Nombre de lignes dans 'segments': 4
✔️ Nombre de lignes dans 'agences': 8
✔️ Nombre de lignes dans 'produits': 8
✔️ Nombre de lignes dans 'clients': 150
✔️ Nombre de lignes dans 'comptes': 150
✔️ Nombre de lignes dans 'transactions': 2000

 Sécurité : Toutes les transactions sont rattachées à des comptes existants.


# Requêtes SQL Analytiques Avancées

Agrégations : total et moyenne des transactions par agence, produit et mois (GROUP BY, HAVING)

In [8]:
query_aggregations = """
SELECT 
    a.nom_agence, 
    p.type_produit, 
    EXTRACT(MONTH FROM t.date_transaction) as mois,
    COUNT(t.transaction_id) as nb_transactions,
    SUM(t.montant_eur) as total_montant,
    ROUND(AVG(t.montant_eur)::numeric, 2) as moyenne_montant
FROM transactions t
JOIN comptes c ON t.compte_id = c.compte_id
JOIN agences a ON c.agence_id = a.agence_id
JOIN produits p ON c.produit_id = p.produit_id
GROUP BY a.nom_agence, p.type_produit, mois
HAVING COUNT(t.transaction_id) > 5  
ORDER BY a.nom_agence, mois;
"""
df_aggregations = pd.read_sql(query_aggregations, engine)
df_aggregations

,nom_agence,type_produit,mois,nb_transactions,total_montant,moyenne_montant
0,Bordeaux-Meriadeck,Credit Consommation,1.0,10,4615.74,461.57
1,Bordeaux-Meriadeck,PEA,1.0,6,-3799.75,-633.29
2,Bordeaux-Meriadeck,PEA,2.0,7,-2089.85,-298.55
3,Bordeaux-Meriadeck,Compte Courant,2.0,7,8864.79,1266.40
4,Bordeaux-Meriadeck,Credit Consommation,3.0,6,2032.96,338.83
...,...,...,...,...,...,...
93,Paris-Centre,PEA,3.0,7,-6498.91,-928.42
94,Paris-Centre,PEA,12.0,7,-14003.36,-2000.48
95,Toulouse-Capitole,Compte Epargne,3.0,6,1492.34,248.72
96,Toulouse-Capitole,Compte Epargne,5.0,6,2684.51,447.42


Sous-requêtes : clients avec un solde inférieur à la moyenne nationale

In [9]:
query_subquery = """
SELECT 
    cl.client_id, 
    cl.score_credit, 
    c.solde_avant
FROM clients cl
JOIN comptes c ON cl.client_id = c.client_id
WHERE c.solde_avant < (SELECT AVG(solde_avant) FROM comptes)
ORDER BY c.solde_avant DESC;
"""
df_low_balance_clients = pd.read_sql(query_subquery, engine)
df_low_balance_clients

,client_id,score_credit,solde_avant
0,CLI0127,NaN,24624.11
1,CLI0136,590.0,24532.64
2,CLI0107,420.0,24184.96
3,CLI0147,812.0,24098.03
4,CLI0101,655.0,23134.95
...,...,...,...
71,CLI0014,673.0,2268.89
72,CLI0128,683.0,1161.31
73,CLI0049,593.0,960.44
74,CLI0013,351.0,684.45


CASE WHEN : calcul du taux de défaut par segment de risque

In [10]:
query_case_when = """
SELECT 
    cl.categorie_risque,
    COUNT(t.transaction_id) as total_trans,
    SUM(CASE WHEN t.is_rejected = TRUE THEN 1 ELSE 0 END) as nb_rejets,
    ROUND(
        (SUM(CASE WHEN t.is_rejected = TRUE THEN 1 ELSE 0 END)::numeric / COUNT(t.transaction_id)) * 100, 2
    ) as taux_defaut_pct
FROM clients cl
JOIN comptes c ON cl.client_id = c.client_id
JOIN transactions t ON c.compte_id = t.compte_id
GROUP BY cl.categorie_risque;
"""
df_risk_analysis = pd.read_sql(query_case_when, engine)
df_risk_analysis

,categorie_risque,total_trans,nb_rejets,taux_defaut_pct
0,Low,444,27,6.08
1,NaN,245,12,4.90
2,High,410,19,4.63
3,Medium,901,55,6.10


Jointures multi-tables

In [11]:
query_multi_joins = """
SELECT 
    t.transaction_id, 
    t.date_transaction, 
    t.montant_eur, 
    cl.client_id, 
    s.nom_segment, 
    a.nom_agence, 
    p.type_produit
FROM transactions t
JOIN comptes c ON t.compte_id = c.compte_id
JOIN clients cl ON c.client_id = cl.client_id
JOIN segments s ON cl.segment_id = s.segment_id
JOIN agences a ON c.agence_id = a.agence_id
JOIN produits p ON c.produit_id = p.produit_id
LIMIT 10;
"""
df_global_view = pd.read_sql(query_multi_joins, engine)
df_global_view

,transaction_id,date_transaction,montant_eur,client_id,nom_segment,nom_agence,type_produit
0,TXN000559,2022-04-19 02:31:00,2050.42,CLI0060,Premium,Marseille-Vieux-Port,Compte Epargne
1,TXN000764,2024-08-28 05:03:00,-396.17,CLI0015,Standard,Lyon-Part-Dieu,PEA
2,TXN001598,2024-01-07 08:16:00,225.20,CLI0045,Standard,Bordeaux-Meriadeck,Credit Consommation
3,TXN001873,2024-08-11 19:52:00,935.32,CLI0034,Risque,Bordeaux-Meriadeck,Credit Immobilier
4,TXN001864,2024-04-28 23:08:00,-264.67,CLI0061,Risque,Marseille-Vieux-Port,Compte Epargne
5,TXN000931,2023-09-13 12:04:00,2752.67,CLI0118,Standard,Lille-Grand-Place,Compte Epargne
6,TXN000620,2022-05-11 08:57:00,1962.49,CLI0005,Standard,Lille-Grand-Place,Livret A
7,TXN000581,2022-10-24 11:30:00,422.70,CLI0064,Premium,Lille-Grand-Place,Compte Courant
8,TXN000726,2022-04-21 21:15:00,6451.32,CLI0085,Risque,Toulouse-Capitole,Credit Consommation
9,TXN000701,2023-11-08 18:40:00,2407.48,CLI0009,Standard,Nice-Massena,Compte Epargne


Créer des vues analytiques pour les KPIs du dashboard

In [12]:
with engine.connect() as conn:
    
    conn.execute(text("""
        CREATE OR REPLACE VIEW v_kpi_performance_agences AS
        SELECT a.nom_agence, SUM(t.montant_eur) as volume_total, COUNT(*) as nb_trans
        FROM transactions t
        JOIN comptes c ON t.compte_id = c.compte_id
        JOIN agences a ON c.agence_id = a.agence_id
        GROUP BY a.nom_agence;
    """))
    
    conn.execute(text("""
        CREATE OR REPLACE VIEW v_kpi_risque_client AS
        SELECT categorie_risque, AVG(score_credit) as avg_score, COUNT(*) as nb_clients
        FROM clients
        GROUP BY categorie_risque;
    """))
    
    conn.commit()
    print("Analytical Views : Generation terminee avec succes.")

Analytical Views : Generation terminee avec succes.
